# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'ahuwer2'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
extinction = load('p1a_extinction')
#print(extinction.head())
ext_x  = extinction['airmass'].to_numpy()
ext_y  = extinction['zeropoint_mag'].to_numpy()
ext_sy = extinction['sigma_quoted_mag'].to_numpy()
ext_N  = len(ext_x)

# Weighted least squares, quoted errors only
ext_A = np.column_stack([ext_x, np.ones(ext_N)])
ext_Sinv = np.diag(1.0 / ext_sy**2)
cov_wls = np.linalg.inv(ext_A.T @ ext_Sinv @ ext_A)
ext_theta = cov_wls @ ext_A.T @ ext_Sinv @ ext_y
k_hat, m0_hat = ext_theta
k_err  = np.sqrt(cov_wls[0, 0])
m0_err = np.sqrt(cov_wls[1, 1])
print(f"WLS:     k  = {k_hat:.5f} +/- {k_err:.5f} mag/airmass")
print(f"         m0 = {m0_hat:.5f} +/- {m0_err:.5f} mag\n")

# np.polyfit with w = 1/sigma and cov=True
coef, cov_pf = np.polyfit(ext_x, ext_y, 1, w=1/ext_sy, cov=True)
print(f"polyfit: k  = {coef[0]:.5f} +/- {np.sqrt(cov_pf[0, 0]):.5f} mag/airmass")
print(f"         m0 = {coef[1]:.5f} +/- {np.sqrt(cov_pf[1, 1]):.5f} mag")
print(f"ratio polyfit err / WLS err on k = {np.sqrt(cov_pf[0, 0]) / k_err:.4f}\n")

# Chi-square, dof, and tail probabilities
ext_resid = ext_y - ext_A @ ext_theta
ext_chi2 = np.sum((ext_resid / ext_sy)**2)
ext_dof  = ext_N - 2
p_large = stats.chi2.sf(ext_chi2, ext_dof)    # P(chi2 >= observed)
p_small = stats.chi2.cdf(ext_chi2, ext_dof)   # P(chi2 <= observed)
print(f"WLS chi2 = {ext_chi2:.3f}, dof = {ext_dof}, chi2/dof = {ext_chi2/ext_dof:.4f}")
print(f"P(chi2 >= {ext_chi2:.3f}) = {p_large:.4g}")
print(f"P(chi2 <= {ext_chi2:.3f}) = {p_small:.4g}")

# Quantities for deciding the quoted error bar
ext_scale = np.sqrt(ext_chi2 / ext_dof)
print(f"sqrt(chi2/dof) = {ext_scale:.4f}")
print(f"k error scaled by sqrt(chi2/dof) = {k_err * ext_scale:.5f}")

# Plot
ext_xx = np.linspace(ext_x.min(), ext_x.max(), 200)
plt.errorbar(ext_x, ext_y, yerr=ext_sy, fmt='o', capsize=3, label='data')
plt.plot(ext_xx, k_hat * ext_xx + m0_hat, label='WLS fit')
plt.xlabel('airmass')
plt.ylabel('zero point (mag)')
plt.legend()
plt.show()

plt.figure()
plt.errorbar(ext_x, ext_resid / ext_sy, yerr=1, fmt='o', capsize=3)
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('airmass')
plt.ylabel('normalized residual')
plt.show()


**ANSWER:**

1. From above, $k=0.18881\pm0.01292$ mag / airmass.
2. Also from above, $k=0.18881\pm0.00522$ mag / airmass using np.polyfit. This difference is due to the fact that, by default, np.polyfit assumes the weights in the covariant matrix are unreliable except in the relative sense, so they scale by $\frac{\chi^2}{\mathrm{DOF}}$. The base WLS code implemented in 1 makes no such assumption, hence the differing errorbars.
3. As seen above, the probability that $\chi^2$ is at least 2.124 (the value given by the matrix approach and not np.polyfit's default behavior) is over 99.9%, indicating that we have an unusually small $\chi^2$ given our data and pipeline error bars. This is backed up by the small value of $\frac{\chi^2}{\mathrm{DOF}}=0.1634$, which should be closer to 1 to confidently indicate that our $\chi^2$ value is consistent with our pipeline errors.
4. Due to 3, I'd quote the numpy polyfit errors over the pipeline errors, which seem to be overconservative with errors significantly larger than we should expect given our results.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# ---- 1b: centroid error vs Cramer-Rao bound ----
centroid = load('p1b_centroid')
#print(centroid.head())

def cen_fisher_per_photon(sig_psf, h=1e-5, npts=200_001, span=8):
    """Numerical Fisher information per photon on the star position x0 (lecture method)."""
    cen_xgrid = np.linspace(-span*sig_psf, span*sig_psf, npts)
    cen_p  = stats.norm.pdf(cen_xgrid, 0.0, sig_psf)
    cen_dp = (stats.norm.pdf(cen_xgrid, h, sig_psf) - stats.norm.pdf(cen_xgrid, -h, sig_psf)) / (2*h)
    return np.trapezoid(cen_dp**2 / cen_p, cen_xgrid)

for cen_i, cen_row in centroid.iterrows():
    cen_N      = cen_row['n_photons']
    cen_sig    = cen_row['psf_sigma_arcsec']
    cen_quoted = cen_row['quoted_sigma_x0_arcsec']

    # Cramer-Rao bound
    cen_I_one    = cen_fisher_per_photon(cen_sig)
    cen_crb_num  = 1.0 / np.sqrt(cen_N * cen_I_one)     # numerical Fisher integral
    cen_crb_simp = cen_sig / np.sqrt(cen_N)             # simple closed form (I per photon = 1/sigma^2)

    print(f"\nrow {cen_i}: N = {cen_N:.0f}, psf_sigma = {cen_sig:.4f} arcsec")
    print(f"  I per photon (numerical) = {cen_I_one:.6g}   1/sigma^2 = {1/cen_sig**2:.6g}")
    print(f"  CRB, numerical Fisher    = {cen_crb_num:.6f} arcsec")
    print(f"  CRB, sigma/sqrt(N)       = {cen_crb_simp:.6f} arcsec")

    # Compare with the quoted error
    print(f"  quoted sigma_x0          = {cen_quoted:.6f} arcsec")
    print(f"  quoted / CRB             = {cen_quoted / cen_crb_simp:.4f}")

**ANSWER:**

The quoted error is *impossible*, as the error is significantly below the Cramér-Rao bound. The Cramér-Rao bound places a numerical limit on the lowest error an unbiased estimator can produce, so if the estimator reports a lower error, it's either an unintentionally biased esitmator, or is intentionally biased, like if the weighting scheme used a prior and the classmate just pasted it without justification. It's also possible they just incorrectly computed the error's actual value, reported variance = $\sigma^2$ instead of $\sigma$, etc.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
# ---- 1c: transit depth profile likelihood ----
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

profile = load('p1c_profile')
#print(profile.columns.tolist())
prof_d = profile['depth_ppt'].to_numpy()
prof_c = profile['minus2lnL'].to_numpy()
prof_order = np.argsort(prof_d)
prof_d, prof_c = prof_d[prof_order], prof_c[prof_order]

# Smooth interpolation of the tabulated profile so we can minimise and differentiate it
prof_f = CubicSpline(prof_d, prof_c)

# Best depth (minimiser), curvature by finite differences at the minimum
prof_i0  = np.argmin(prof_c)
prof_res = minimize(lambda m: prof_f(m[0]), x0=[prof_d[prof_i0]], bounds=[(prof_d.min(), prof_d.max())])
prof_best = prof_res.x[0]
prof_cmin = float(prof_f(prof_best))

def prof_second_deriv(h):
    return (prof_f(prof_best + h) - 2*prof_f(prof_best) + prof_f(prof_best - h)) / h**2

prof_h = 1e-3 * np.ptp(prof_d)
prof_d2 = float(prof_second_deriv(prof_h))     # d^2(-2lnL)/d depth^2
print(f"best depth = {prof_best:.5f} ppt,  min(-2lnL) = {prof_cmin:.5f},  h = {prof_h:.2e}")

# curvature C of -lnL is half of d2 of -2lnL
prof_C = 0.5 * prof_d2
prof_sig_correct = 1 / np.sqrt(prof_C)         # sigma = 1/sqrt(C)
prof_sig_class   = 1 / np.sqrt(prof_d2)        # classmate: 1/sqrt(d2 of -2lnL)
print(f"d2(-2lnL)/dd2 = {prof_d2:.5f},  C = {prof_C:.5f}")
print(f"\nCorrect sigma   = {prof_sig_correct:.5f} ppt")
print(f"Classmate sigma = {prof_sig_class:.5f} ppt")
print(f"Classmate / Correct = {prof_sig_class/prof_sig_correct:.4f},  Correct / Classmate = {prof_sig_correct/prof_sig_class:.4f}")

# Delta(-2lnL) = 1 interval read off the profile
prof_g = lambda x: float(prof_f(x)) - prof_cmin - 1.0
try:
    prof_lo = brentq(prof_g, prof_d.min(), prof_best)
except ValueError:
    prof_lo = np.nan
try:
    prof_hi = brentq(prof_g, prof_best, prof_d.max())
except ValueError:
    prof_hi = np.nan
prof_m = prof_best - prof_lo          # minus-side half-width of Delta=1 interval
prof_p = prof_hi - prof_best          # plus-side half-width
prof_s = prof_sig_correct             # curvature sigma

prof_avg  = 0.5 * (prof_m + prof_p)   # symmetric (even) part
prof_asym = 0.5 * (prof_m - prof_p)   # asymmetric (odd) part

print(f"Delta(-2lnL)=1 interval: {prof_best:.4f} +{prof_p:.4f} / -{prof_m:.4f} ppt")
print(f"Curvature sigma:         {prof_best:.4f} +/- {prof_s:.4f} ppt\n")

print("Each side vs curvature sigma (fractional difference):")
print(f"  minus side: {prof_m:.5f} vs {prof_s:.5f}  -> {100*(prof_m/prof_s - 1):+.2f}%")
print(f"  plus side:  {prof_p:.5f} vs {prof_s:.5f}  -> {100*(prof_p/prof_s - 1):+.2f}%\n")

print("Splitting the interval into symmetric and asymmetric parts:")
print(f"  average half-width = {prof_avg:.5f}  -> {100*(prof_avg/prof_s - 1):+.2f}% from curvature sigma")
print(f"  asymmetry (m-p)/2  = {prof_asym:.5f}  -> {100*prof_asym/prof_avg:.2f}% of the half-width")
print(f"  plus/minus ratio   = {prof_p/prof_m:.4f}\n")

# 3. Coverage of classmate's +/- sigma
prof_k = prof_sig_class / prof_sig_correct
prof_pred = stats.norm.cdf(prof_k) - stats.norm.cdf(-prof_k)
print(f"Predicted coverage of classmate interval: P(|Z| < {prof_k:.4f}) = {prof_pred:.4f}")

prof_rng   = np.random.default_rng(20260909)
prof_truth = prof_best                 # chosen truth
prof_nsim  = 5000
prof_fits  = prof_rng.normal(prof_truth, prof_sig_correct, prof_nsim)
prof_cov_class = np.mean(np.abs(prof_fits - prof_truth) < prof_sig_class)
prof_cov_corr  = np.mean(np.abs(prof_fits - prof_truth) < prof_sig_correct)
prof_se = lambda p: np.sqrt(p*(1-p)/prof_nsim)
print(f"Simulation ({prof_nsim} fake fits, truth = {prof_truth:.4f}):")
print(f"  classmate +/-sigma coverage = {prof_cov_class:.4f} +/- {prof_se(prof_cov_class):.4f}")
print(f"  correct   +/-sigma coverage = {prof_cov_corr:.4f} +/- {prof_se(prof_cov_corr):.4f}")

# Plots
prof_xx = np.linspace(prof_d.min(), prof_d.max(), 1000)
fig, prof_ax = plt.subplots(1, 3, figsize=(17, 4.5))
for prof_a, prof_zoom in zip(prof_ax[:2], [False, True]):
    prof_a.plot(prof_d, prof_c, 'k.', ms=4, label='profile')
    prof_a.plot(prof_xx, prof_f(prof_xx), 'k-', lw=0.8)
    prof_a.plot(prof_xx, prof_cmin + (prof_xx - prof_best)**2 / prof_sig_correct**2, 'C3--', label='parabola, correct curvature')
    prof_a.plot(prof_xx, prof_cmin + (prof_xx - prof_best)**2 / prof_sig_class**2, 'C0--', label='parabola, classmate')
    prof_a.axhline(prof_cmin + 1, color='gray', ls=':', label=r'min + 1')
    prof_a.axvspan(prof_best - prof_sig_correct, prof_best + prof_sig_correct, color='C3', alpha=0.15, label=r'correct $\pm\sigma$')
    prof_a.axvspan(prof_best - prof_sig_class, prof_best + prof_sig_class, color='C0', alpha=0.15, label=r'classmate $\pm\sigma$')
    prof_a.axvline(prof_best, color='C2', ls=':')
    prof_a.plot([prof_lo, prof_hi], [prof_cmin + 1]*2, 'm|', ms=20, mew=2, label=r'$\Delta(-2\ln L)=1$ ends')
    prof_a.set_xlabel('depth (ppt)')
    prof_a.set_ylabel(r'$-2\ln L$')
    if prof_zoom:
        prof_w = 3 * prof_sig_correct
        prof_a.set_xlim(prof_best - prof_w, prof_best + prof_w)
        prof_a.set_ylim(prof_cmin - 0.5, prof_cmin + 6)
        prof_a.set_title('zoom near minimum')
    else:
        prof_a.set_title('full profile')
        prof_a.legend(fontsize=7)

prof_ax[2].hist(prof_fits - prof_truth, bins=50, color='gray', alpha=0.6)
prof_ax[2].axvspan(-prof_sig_correct, prof_sig_correct, color='C3', alpha=0.15, label=r'correct $\pm\sigma$')
prof_ax[2].axvspan(-prof_sig_class, prof_sig_class, color='C0', alpha=0.15, label=r'classmate $\pm\sigma$')
prof_ax[2].set_xlabel('fit - truth (ppt)')
prof_ax[2].set_title('simulated best-fit depths')
prof_ax[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

**ANSWER:**

Their errorbar is incorrect and is small by a factor of approximately 0.7071, which happens to be $1/\sqrt{2}$. I'm almost positive that is the result of reporting $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$ rather than $\sigma = 1/\sqrt{d^2(-\ln L)/d\,\textrm{depth}^2}$, which would result in the errors being scaled by exactly $1/\sqrt{2}$ due to using $-2\ln L$ instead of $-\ln L$. The $\Delta(-2\ln L)=1$ interval is very slightly asymmetric, with the positive side being only 96% as wide as the negative side. Compared to the curvature value of error, the delta values are about 2% larger in the negative width and 2% smaller in positive width.

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
mass = load('p1d_mass_samples')
mass_m = mass['M2_Msun'].to_numpy()
mass_N = len(mass_m)

# Median with 16th-84th percentile error bars
mass_p16, mass_med, mass_p84 = np.percentile(mass_m, [16, 50, 84])
mass_lo, mass_hi = mass_med - mass_p16, mass_p84 - mass_med
mass_ref = 0.5 * (mass_p84 - mass_p16)          # reference 1-sigma error bar (half the 68% interval)
print(f"M2 = {mass_med:.4f} +{mass_hi:.4f} -{mass_lo:.4f} Msun   (reference 1-sigma = {mass_ref:.4f} Msun)")

# The three classmates' numbers and their ratio to the reference 1-sigma
mass_std = np.std(mass_m, ddof=1)
mass_claims = {
    '(i) variance':      (np.var(mass_m, ddof=1),            'Msun^2'),
    '(ii) p84 - p16':    (mass_p84 - mass_p16,               'Msun'),
    '(iii) std/sqrt(N)': (mass_std / np.sqrt(mass_N),        'Msun'),
}
mass_ratios = {}
for mass_k, (mass_v, mass_u) in mass_claims.items():
    mass_ratios[mass_k] = mass_v / mass_ref          # units stripped for (i)
    print(f"{mass_k:<18} = {mass_v:.5g} {mass_u:<7} -> ratio to 1-sigma = {mass_ratios[mass_k]:.4g}")
print(f"(for scale: 1/sqrt(N) = {1/np.sqrt(mass_N):.4f}, std/half-width = {mass_std/mass_ref:.3f})")

# Symmetry check
print(f"minus = {mass_lo:.4f}, plus = {mass_hi:.4f}, plus/minus = {mass_hi/mass_lo:.3f}, skewness = {stats.skew(mass_m):.3f}")

# Plots
fig, mass_ax = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) posterior with asymmetric vs symmetric error bars
mass_a = mass_ax[0]
mass_a.hist(mass_m, bins=80, color='gray', alpha=0.6, density=True)
mass_a.axvline(mass_med, color='k', label='median')
mass_a.axvspan(mass_p16, mass_p84, color='C3', alpha=0.15, label='16th-84th percentile')
mass_a.axvspan(mass_med - mass_ref, mass_med + mass_ref, color='C0', alpha=0.15, label='median $\\pm$ half-width (symmetric)')
mass_a.set_xlabel(r'$M_2$ ($M_\odot$)')
mass_a.legend(fontsize=8)

# (b) ratio of each claimed number to the true 1-sigma (log scale; 1 = correct)
mass_b = mass_ax[1]
mass_names = list(mass_ratios.keys())
mass_vals = [mass_ratios[k] for k in mass_names]
mass_b.bar(mass_names, mass_vals, color=['C1', 'C2', 'C4'])
mass_b.axhline(1, color='k', ls='--', label='1-sigma')
mass_b.set_yscale('log')
mass_b.set_ylabel('claimed / reference (std) 1-sigma')
mass_b.legend(fontsize=8)
plt.tight_layout()
plt.show()

**ANSWER:**

The exact values are shown above, with correct units. Specifically, (i) the variance measures how far the values are spread from the mean, and is the square of the standard deviation (which we know to have the same units as the base quantity, so the variance's units here are $M_\odot^2$). The 84th percentile minus the 16th is a measure of the 68% width of the distribution with units in $M_\odot$, very similar to the standard deviation except it is a bit more robust as the percentiles work better for skewed distributions, such as the one in this problem. The $\sigma/\sqrt{N}$ case has the units of $M_\odot$ and shouldn't be used to represent the error of the distribution but rather the error of the reported mean to the true mean given our number of data points. Here we can see the ratios to $1\sigma$ are all off: the variance first isn't even the correct value or units to report as an errorbar, and is thus off as it's the square of the $1\sigma$ deviation. The percentile case is $2\sigma$ in width, which is actually what you'd expect given that it should be measuring $1\sigma$ in either direction. It's the best one to report as long as they don't state the whole width is a $\pm$ errorbar and instead use $^{+(P_{84}-P_{50})}_{-(P_{50}-P_{16})}$. The $\sigma/\sqrt{N}$ case is obviously off by $1/\sqrt{N}$. Here we can see from the percentile case that the errorbars are not symmetric, which makes sense as the distribution is clearly somewhat skewed, so reporting symmetric $\pm$ error bars is not a good choice.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# ---- 1e: slope at fixed intercept vs marginal ----
si = load('p1e_slope_intercept_samples')
#print(si.columns.tolist())
si_s = si.iloc[:, 0].to_numpy()
si_b = si.iloc[:, 1].to_numpy()

# 1. Marginal error and fixed-intercept (conditional) error
si_cov  = np.cov(si_s, si_b)
si_marg = np.sqrt(si_cov[0, 0])
si_fix  = 1 / np.sqrt(np.linalg.inv(si_cov)[0, 0])     # 1-D fit at fixed intercept: 1/sqrt(precision of slope)
si_rho  = si_cov[0, 1] / np.sqrt(si_cov[0, 0] * si_cov[1, 1])
si_s0, si_b0 = si_s.mean(), si_b.mean()

# direct check from the samples: slope spread in a thin slice around the best intercept
si_slice = np.abs(si_b - si_b0) < 0.1 * si_b.std()
si_fix_slice = si_s[si_slice].std(ddof=1)

print(f"marginal  sigma_slope = {si_marg:.5f}")
print(f"fixed-int sigma_slope = {si_fix:.5f}   (slice check: {si_fix_slice:.5f}, n = {si_slice.sum()})")
print(f"difference (marginal - fixed) = {si_marg - si_fix:.5f}")
print(f"ratio fixed/marginal = {si_fix/si_marg:.4f},  rho = {si_rho:.4f},  sqrt(1-rho^2) = {np.sqrt(1 - si_rho**2):.4f}")

# 2. Plots
fig, si_ax = plt.subplots(1, 3, figsize=(17, 4.8))

# (a) samples with both error bars on the slope axis
si_a = si_ax[0]
si_a.plot(si_b, si_s, '.', ms=2, alpha=0.3, color='gray')
si_a.plot(si_b0, si_s0, 'k+', ms=12)
si_a.axhspan(si_s0 - si_marg, si_s0 + si_marg, color='C3', alpha=0.15, label=r'marginal $\pm\sigma$')
si_a.axhspan(si_s0 - si_fix,  si_s0 + si_fix,  color='C0', alpha=0.25, label=r'fixed-intercept $\pm\sigma$')
si_a.errorbar(si_b0, si_s0, yerr=si_marg, color='C3', capsize=5, lw=2)
si_a.errorbar(si_b0, si_s0, yerr=si_fix,  color='C0', capsize=5, lw=2)
si_a.set_xlabel('intercept')
si_a.set_ylabel('slope')
si_a.set_title(f'samples, rho = {si_rho:.3f}')
si_a.legend(fontsize=8)

# (b) the two slope distributions against each other
si_a = si_ax[1]
si_bins = np.linspace(si_s.min(), si_s.max(), 80)
si_a.hist(si_s, bins=si_bins, density=True, color='C3', alpha=0.35, label=f'marginal ({si_marg:.4f})')
si_xx = np.linspace(si_s.min(), si_s.max(), 500)
si_a.plot(si_xx, stats.norm.pdf(si_xx, si_s0, si_fix), 'C0', lw=2, label=f'fixed intercept ({si_fix:.4f})')
si_a.hist(si_s[si_slice], bins=si_bins, density=True, histtype='step', color='C0', ls='--', label='thin slice in intercept')
si_a.set_xlabel('slope')
si_a.set_title('slope: marginal vs fixed intercept')
si_a.legend(fontsize=8)

# (c) ratio vs rho
si_a = si_ax[2]
si_rr = np.linspace(-1, 1, 400)
si_a.plot(si_rr, np.sqrt(1 - si_rr**2), 'k-', label=r'$\sqrt{1-\rho^2}$')
si_a.plot(si_rho, si_fix / si_marg, 'ro', label='this data')
si_a.set_xlabel(r'$\rho$')
si_a.set_ylabel('fixed / marginal error')
si_a.legend(fontsize=8)

plt.tight_layout()
plt.show()

**ANSWER:**

The values asked for are shown above. Here we can see that the classmate's choice of error is incorrect here, specifically too small. Their error would be correct to quote if we had a known value for the best intercept, but in this case that was only determined from the fit, so treating the slope errors as if the intercept is actually known is not correct.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
from astropy.cosmology import FlatLambdaCDM

sn = load('p2_pantheon_sne')
#print(sn.head())
sn_z    = sn['z'].to_numpy()
sn_mB   = sn['mB'].to_numpy()
sn_x1   = sn['x1'].to_numpy()
sn_c    = sn['c'].to_numpy()
sn_mass = sn['host_logmass'].to_numpy()
sn_N    = len(sn_z)

SN_CLIP = 7.5

# Distance modulus and Hubble residuals
sn_mu  = FlatLambdaCDM(H0=70, Om0=0.3).distmod(sn_z).value
sn_res = sn_mB - sn_mu

# Clipping model (visual aid only): res = a*x1 + b*c + d, least squares with iterative clipping
sn_A = np.column_stack([sn_x1, sn_c, np.ones(sn_N)])
sn_keep = np.ones(sn_N, bool)
for _ in range(2):
    sn_coef = np.linalg.lstsq(sn_A[sn_keep], sn_res[sn_keep], rcond=None)[0]
    sn_dev  = sn_res - sn_A @ sn_coef
    sn_mad  = 1.4826 * np.median(np.abs(sn_dev[sn_keep] - np.median(sn_dev[sn_keep])))
    sn_keep = np.abs(sn_dev) < SN_CLIP * sn_mad
sn_out = np.where(~sn_keep)[0]
sn_a_, sn_b_, sn_d_ = sn_coef
sn_band = SN_CLIP * sn_mad

print(f"clipping curve: m_B - mu = {sn_a_:+.4f}*x1 {sn_b_:+.4f}*c {sn_d_:+.4f}   (robust sigma = {sn_mad:.4f} mag)")
print(f"clip at {SN_CLIP} sigma = +/-{sn_band:.3f} mag about the curve -> {len(sn_out)} flagged")
print(f"corr(x1, host_logmass) = {np.corrcoef(sn_x1, sn_mass)[0, 1]:+.3f}")
print("flagged (index, residual, deviation in sigma):")
for sn_i in sn_out:
    print(f"  {sn_i:4d}  {sn_res[sn_i]:.3f}  {sn_dev[sn_i]/sn_mad:+.1f}")

# Labels
sn_lbl_res  = r'$m_B - \mu(z)$ [mag]'
sn_lbl_x1   = r'$x_1$'
sn_lbl_c    = r'$c$'
sn_lbl_mass = r'$\log_{10}(M_\star/M_\odot)$'

# Plots
fig, sn_ax = plt.subplots(3, 3, figsize=(16, 12))

# ---- Row 0: the plain plots the problem asks for (all points, simple linear fit, no clipping) ----
for sn_a, sn_xv, sn_xl in [(sn_ax[0, 0], sn_x1, sn_lbl_x1),
                           (sn_ax[0, 1], sn_c, sn_lbl_c),
                           (sn_ax[0, 2], sn_mass, sn_lbl_mass)]:
    sn_a.plot(sn_xv, sn_res, 'k.', ms=4)
    sn_p = np.polyfit(sn_xv, sn_res, 1)
    sn_xx = np.linspace(sn_xv.min(), sn_xv.max(), 100)
    sn_a.plot(sn_xx, np.polyval(sn_p, sn_xx), 'C0-', label=rf'linear fit, slope ${sn_p[0]:+.3f}$')
    if sn_xv is sn_mass:
        sn_a.axvline(10, color='gray', ls=':')
    sn_a.set_xlabel(sn_xl)
    sn_a.set_ylabel(sn_lbl_res)
    sn_a.legend(fontsize=8)

# ---- Row 1: x1 vs host mass (asked in part 3), sorted deviations, deviation vs host mass ----
sn_a = sn_ax[1, 0]
sn_a.plot(sn_mass, sn_x1, 'k.', ms=4)
sn_p = np.polyfit(sn_mass, sn_x1, 1)
sn_xx = np.linspace(sn_mass.min(), sn_mass.max(), 100)
sn_a.plot(sn_xx, np.polyval(sn_p, sn_xx), 'C0-', label=rf'linear fit, slope ${sn_p[0]:+.3f}$')
sn_a.axvline(10, color='gray', ls=':')
sn_a.set_xlabel(sn_lbl_mass)
sn_a.set_ylabel(sn_lbl_x1)
sn_a.legend(fontsize=8)

sn_a = sn_ax[1, 1]
sn_sorted = np.sort(np.abs(sn_dev) / sn_mad)[::-1]
sn_a.plot(np.arange(1, 21), sn_sorted[:20], 'ko-', ms=4)
for sn_t in [4, 5, 7.5]:
    sn_a.axhline(sn_t, color='gray', ls=':')
    sn_a.text(20, sn_t, rf' ${sn_t:g}\sigma$', va='bottom', ha='right', fontsize=8)
sn_a.set_xlabel('rank (largest first)')
sn_a.set_ylabel(r'$|\Delta|\,/\,\sigma_{\rm robust}$')
sn_a.set_title('20 largest deviations from clipping curve')

sn_a = sn_ax[1, 2]
sn_a.plot(sn_mass, sn_dev, 'k.', ms=4)
sn_a.plot(sn_mass[sn_out], sn_dev[sn_out], 'ro', mfc='none', ms=9, label='flagged')
sn_a.axhspan(-sn_band, sn_band, color='C0', alpha=0.15, label=rf'$\pm{SN_CLIP:g}\sigma$')
sn_a.axvline(10, color='gray', ls=':')
sn_a.set_xlabel(sn_lbl_mass)
sn_a.set_ylabel(r'deviation from clipping curve [mag]')
sn_a.legend(fontsize=8)

# ---- Row 2: partial-residual plots showing the actual clipping curve ----
for sn_a, sn_xv, sn_part, sn_slope, sn_xl, sn_other in [
        (sn_ax[2, 0], sn_x1, sn_res - sn_b_*sn_c,  sn_a_, sn_lbl_x1, r'$- b\,c$'),
        (sn_ax[2, 1], sn_c,  sn_res - sn_a_*sn_x1, sn_b_, sn_lbl_c,  r'$- a\,x_1$')]:
    sn_xx = np.linspace(sn_xv.min(), sn_xv.max(), 100)
    sn_ln = sn_slope*sn_xx + sn_d_
    sn_a.plot(sn_xv, sn_part, 'k.', ms=4)
    sn_a.plot(sn_xv[sn_out], sn_part[sn_out], 'ro', mfc='none', ms=9, label='flagged')
    sn_a.plot(sn_xx, sn_ln, 'C0-', label=rf'clipping curve, slope ${sn_slope:+.3f}$')
    sn_a.fill_between(sn_xx, sn_ln - sn_band, sn_ln + sn_band, color='C0', alpha=0.15,
                      label=rf'$\pm{SN_CLIP:g}\sigma$ (${sn_band:.3f}$ mag)')
    sn_a.set_xlabel(sn_xl)
    sn_a.set_ylabel(r'$m_B - \mu(z)$ ' + sn_other + ' [mag]')
    sn_a.legend(fontsize=8)

sn_ax[2, 2].axis('off')
plt.tight_layout()
plt.show()

**ANSWER:**

The $x_1$ trend is slightly negative, and the $c$ trend is positive, both of which match the expected signs in the Tripp relation. Specifically (with only an intentionally basic linear fit), we find $m_B - \mu(z)$ depends on $x_1$ with factor $-\alpha=-0.108$, and dependent on $c$ with factor $+\beta=+2.687$. Here the $x_1(\log(M_\star))$ plot shows that we have some general (seemingly negative) correlation between $x_1$ and $\log(M_\star)$ and thus if you change the host mass, there will also be a change in the $x_1$ term at least (and potentially the $c$ term too, if there's correlation there as well). Because of this correlation, gamma might not actually be positive.

As for contaminants, the last 4 plots show that there are 4 points which are significantly outside the region we would expect, with deviations much than the next 16 highest outliers. These are highly likely to be the added contaminats.

Based on the info given about $x_1$ and $c$, which are both unitless, we must have that $\alpha$ and $\beta$ are in units of mag.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here

# --- likelihood ---
lk_cl   = 299792.458   # km/s
lk_z    = sn['z'].to_numpy()
lk_mB   = sn['mB'].to_numpy()
lk_x1   = sn['x1'].to_numpy()
lk_c    = sn['c'].to_numpy()
lk_mass = sn['host_logmass'].to_numpy()
lk_Cmm  = sn['mB_err'].to_numpy()**2
lk_Cxx  = sn['x1_err'].to_numpy()**2
lk_Ccc  = sn['c_err'].to_numpy()**2
lk_Cmx  = sn['cov_mB_x1'].to_numpy()
lk_Cmc  = sn['cov_mB_c'].to_numpy()
lk_Cxc  = sn['cov_x1_c'].to_numpy()
lk_sig_pec2 = (5/np.log(10) * 250/(lk_cl*lk_z))**2
lk_step = (lk_mass > 10).astype(float)
lk_res0 = lk_mB - sn_mu

def lk_loglike(theta, mask=None):
    """theta = (M, alpha, beta, gamma, ln_sigma_int)"""
    M, alpha, beta, gamma, lnsig = theta
    r = lk_res0 - M + alpha*lk_x1 - beta*lk_c - gamma*lk_step
    vCv = (lk_Cmm + alpha**2*lk_Cxx + beta**2*lk_Ccc
           + 2*alpha*lk_Cmx - 2*beta*lk_Cmc - 2*alpha*beta*lk_Cxc)
    var = vCv + lk_sig_pec2 + np.exp(2*lnsig)
    ll = -0.5*(r**2/var + np.log(var) + np.log(2*np.pi))
    return ll.sum() if mask is None else ll[mask].sum()

# --- contaminants: drop the points flagged by the 2a clipping ---
lk_mask = sn_keep.copy()
print(f"using {lk_mask.sum()} of {len(lk_mask)} SNe ({(~lk_mask).sum()} removed)")

# --- flat priors in (M, alpha, beta, gamma, ln_sigma_int); box bounds only, no Jacobian ---
pr_bounds = {
    'M':     (sn_d_ - 3.0, sn_d_ + 3.0),      # centered on the 2a plane intercept
    'alpha': (-2.0, 2.0),                     # symmetric about 0
    'beta':  (-10.0, 10.0),                   # symmetric about 0
    'gamma': (-1.0, 1.0),                     # symmetric about 0
    'lnsig': (np.log(1e-3), 0.0),             # sigma_int in [1e-3, 1] mag
}
pr_names = list(pr_bounds.keys())
pr_lo = np.array([pr_bounds[k][0] for k in pr_names])
pr_hi = np.array([pr_bounds[k][1] for k in pr_names])

def pr_logprior(theta):
    theta = np.asarray(theta)
    if np.any(theta < pr_lo) or np.any(theta > pr_hi):
        return -np.inf
    return 0.0

def pr_logpost(theta):
    lp = pr_logprior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = lk_loglike(theta, mask=lk_mask)
    return lp + ll if np.isfinite(ll) else -np.inf

# sanity check: posterior is finite at the 2a plane (alpha = -a, beta = b, M = d)
#print(f'Is the posterior finite at the 2a plane? {np.isfinite(pr_logpost([sn_d_, -sn_a_, sn_b_, 0.0, np.log(0.1)]))}')

**ANSWER:**

**Part 1:** Given $v = (1,\alpha,-\beta),$ and the covariance matrix $C_i$ (which by definition is the matrix below),
$$
C_i =
\begin{pmatrix}
\sigma_{m_B,i}^2 & \mathrm{Cov}(m_B,x_1)_i & \mathrm{Cov}(m_B,c)_i \\
\mathrm{Cov}(m_B,x_1)_i & \sigma_{x_1,i}^2 & \mathrm{Cov}(x_1,c)_i \\
\mathrm{Cov}(m_B,c)_i & \mathrm{Cov}(x_1,c)_i & \sigma_{c,i}^2
\end{pmatrix},
$$
we get that
$$
v^T C_i v=\sum_{j,k} v_j\, C_{i,jk}\, v_k.
$$
Expanding this gives
$$
v^T C_i v = \sigma_{m_B,i}^2 + \alpha^2 \sigma_{x_1,i}^2 + \beta^2 \sigma_{c,i}^2 + 2\alpha\,\mathrm{Cov}(m_B,x_1)_i - 2\beta\,\mathrm{Cov}(m_B,c)_i - 2\alpha\beta\,\mathrm{Cov}(x_1,c)_i.
$$
The off-diagonal terms appear in $C_i$ because $m_B$, $x_1$, and $c$ are likely going to be correlated measurements (in fact, we saw some of that correlation earlier in problem 2), so their covariances must be accounted for in the Voc matrix. And since they appear in the matrix, they'll obviously also contribute to the variance.

**Part 2:** Treating the supernovae as independent Gaussians, the likelihood is
$$
\mathcal L(\theta)
=
\prod_i
\frac{1}{\sqrt{2\pi\sigma_i^2}}
\exp\!\left(
-\frac{r_i^2}{2\sigma_i^2}
\right).
$$
with total variance
$$
\sigma_i^2(\alpha,\beta,\sigma_{\rm int}) = \mathbf{v}^T C_i \mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2
$$
and residual
$$
r_i = m_{B,i} - \mu(z_i) - M + \alpha\, x_{1,i} - \beta\, c_i - \gamma\,\Theta(\log M_{\star,i}-10).
$$
Then you just take the log to get the log likelihood, which for this model is given by
$$
\ln \mathcal{L}(\theta) = -\frac{1}{2}\sum_{i=1}^{N}\left[\frac{r_i^2}{\sigma_i^2} + \ln\sigma_i^2 + \ln 2\pi\right].
$$

**Part 3/4** For priors I've chosen uniform priors in $M,\ \alpha, \ \beta, \ \gamma,$ and $\log(\sigma_\mathrm{int})$: $M$ is uniform within a very large range of $\pm3$ mag, to make sure we absolutely don't miss any reasonable values of $M$. I chose $-2<\alpha<2$ and $-10<\beta<10$ to be significantly wider than the values given by the basic linear fits above, which won't be terribly accurate but give a decent idea of the magnitudes. I expect $\gamma$ to be small if not near-zero given how small the published value is, but for safety I allow it to range between -1 and 1 as I'm not sure exactly what magnitude would be considered physically reasonable to use as a bound here, just that $\pm$1 mag should be more than large enough. Meanwhile I made the prior for $\sigma_\mathrm{int}$ uniform in its log, with $10^{-3}<\sigma_\mathrm{int}<1$, as the intrinsic scatter should be positive and relatively small.

**Part 5:** I've decided to simply remove the contaminants: only 4 data points are bad and by over a mag when $\gamma$ is well below 1 mag and $\sigma_\mathrm{int}$ is expected to be below 1 mag. This may cost a couple percent of added error as we eliminate 4/250 points, and technically could risk added bias as we aren't 100% certain the selected points are all contaminants. However, the plots above and the info that at max 4 points were contaminated make it highly likely no real outliers are mistaken for contaminants. They have already been detected and removed with iterative k-sigma clipping, which can all be seen in 2a. I made sure to use the MAD method for the scatter to ensure the contaminant values did not affect the sigma value used to detect contaminants.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
import emcee, corner
import arviz as az
import pandas as pd

mc_names  = ['M', 'alpha', 'beta', 'gamma', 'lnsig']
mc_labels = [r'$M$', r'$\alpha$', r'$\beta$', r'$\gamma$', r'$\ln\sigma_{\rm int}$']
mc_ndim   = 5
mc_rng    = np.random.default_rng(20261013)

def mc_nlp(theta):
    lp = pr_logpost(theta)
    return 1e10 if not np.isfinite(lp) else -lp

# ---- MAP: Nelder-Mead from several starts, then polish ----
mc_x0 = np.array([sn_d_, -sn_a_, sn_b_, 0.0, np.log(0.1)])
mc_best = None
for mc_k in range(6):
    mc_start = mc_x0 if mc_k == 0 else mc_x0 + mc_rng.normal(0, [0.05, 0.05, 0.3, 0.05, 0.3])
    mc_r = minimize(mc_nlp, mc_start, method='Nelder-Mead',
                    options={'xatol': 1e-8, 'fatol': 1e-10, 'maxiter': 20000, 'maxfev': 20000})
    mc_r = minimize(mc_nlp, mc_r.x, method='Nelder-Mead',
                    options={'xatol': 1e-9, 'fatol': 1e-12, 'maxiter': 20000, 'maxfev': 20000})
    if mc_best is None or mc_r.fun < mc_best.fun:
        mc_best = mc_r
mc_map = mc_best.x
print("MAP: " + ", ".join(f"{n} = {v:.4f}" for n, v in zip(mc_names, mc_map)) +
      f"   (sigma_int = {np.exp(mc_map[4]):.4f})")

# ---- posterior width scale from a finite-difference Hessian at the MAP (used only to place starts) ----
def mc_hessian(f, x, h):
    n = len(x); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei, ej = np.eye(n)[i]*h[i], np.eye(n)[j]*h[j]
            H[i, j] = (f(x+ei+ej) - f(x+ei-ej) - f(x-ei+ej) + f(x-ei-ej)) / (4*h[i]*h[j])
    return H
mc_H = mc_hessian(mc_nlp, mc_map, np.array([5e-3, 5e-3, 2e-2, 5e-3, 2e-2]))
try:
    mc_sig = np.sqrt(np.diag(np.linalg.inv(mc_H)))
    assert np.all(np.isfinite(mc_sig))
except Exception:
    mc_sig = np.array([0.02, 0.05, 0.3, 0.03, 0.3])

# ---- starting points: each chain's centre is MAP + 5 sigma * N(0,1) per parameter (wider than the posterior) ----
MC_NCHAIN, MC_NWALK, MC_NSTEP, MC_BURN = 4, 32, 10000, 600
def mc_inside(x):
    return np.all(x > pr_lo) and np.all(x < pr_hi) and np.isfinite(pr_logpost(x))

mc_p0 = []
for mc_c in range(MC_NCHAIN):
    while True:
        mc_centre = mc_map + 5*mc_sig*mc_rng.normal(size=mc_ndim)
        if mc_inside(mc_centre):
            break
    mc_w = []
    while len(mc_w) < MC_NWALK:
        mc_try = mc_centre + 0.1*mc_sig*mc_rng.normal(size=mc_ndim)
        if mc_inside(mc_try):
            mc_w.append(mc_try)
    mc_p0.append(np.array(mc_w))

# ---- run four independent ensembles ----
mc_chains = []     # each (nstep, nwalk, ndim)
for mc_c in range(MC_NCHAIN):
    mc_s = emcee.EnsembleSampler(MC_NWALK, mc_ndim, pr_logpost)
    mc_s.run_mcmc(mc_p0[mc_c], MC_NSTEP, progress=False)
    mc_chains.append(mc_s.get_chain())
    mc_tau = mc_s.get_autocorr_time(tol=0)
    print(f"ensemble {mc_c}: max tau = {mc_tau.max():.0f} steps, acceptance = {mc_s.acceptance_fraction.mean():.2f}")

# ---- Rhat and ESS ----
mc_post_4 = {n: np.stack([c[MC_BURN:, 0, i] for c in mc_chains]) for i, n in enumerate(mc_names)}
mc_sum4 = az.summary(az.from_dict({'posterior': mc_post_4}))
print(mc_sum4[['r_hat', 'ess_bulk']].round(3))

# Trace plots: walker 0 of each independent ensemble, full run (left) and first 600 steps (right)
fig, mc_ax = plt.subplots(5, 2, figsize=(13, 11), sharex='col')
for mc_i, mc_lab in enumerate(mc_labels):
    for mc_c in range(MC_NCHAIN):
        for mc_col, mc_n in enumerate([MC_NSTEP, 600]):
            mc_ax[mc_i, mc_col].plot(mc_chains[mc_c][:mc_n, 0, mc_i], lw=0.6, label=f'chain {mc_c}')
    mc_ax[mc_i, 0].axvline(MC_BURN, color='k', ls='--', lw=1)
    mc_ax[mc_i, 0].set_ylabel(mc_lab)
mc_ax[0, 0].legend(fontsize=7, ncol=4)
mc_ax[0, 0].set_title(r'full run (dashed line = end of burn-in)')
mc_ax[0, 1].set_title('first 600 steps (burn-in)')
mc_ax[-1, 0].set_xlabel('step')
mc_ax[-1, 1].set_xlabel('step')
plt.tight_layout()
plt.show()

# ---- Corner plot and quoted values ----
mc_tau_all = max(emcee.autocorr.integrated_time(c[MC_BURN:], tol=0).max() for c in mc_chains)
mc_thin = max(1, int(mc_tau_all/2))
mc_flat = np.concatenate([c[MC_BURN::mc_thin].reshape(-1, mc_ndim) for c in mc_chains])
mc_sigint = np.exp(mc_flat[:, 4])
print(f"\n{len(mc_flat)} thinned samples (thin = {mc_thin})")

mc_fig = corner.corner(mc_flat, labels=mc_labels, quantiles=[0.16, 0.5, 0.84], show_titles=True,
                       title_fmt='.3f', truths=mc_map)
plt.show()

def mc_quote(x):
    lo, med, hi = np.percentile(x, [16, 50, 84])
    return f"{med:.4f} +{hi-med:.4f} -{med-lo:.4f}"
print("alpha     =", mc_quote(mc_flat[:, 1]))
print("beta      =", mc_quote(mc_flat[:, 2]))
print("gamma     =", mc_quote(mc_flat[:, 3]))
print("sigma_int =", mc_quote(mc_sigint), "mag")
print("edge check (min/max of samples vs box):")
for i, n in enumerate(mc_names):
    print(f"  {n:<6} samples [{mc_flat[:, i].min():.3f}, {mc_flat[:, i].max():.3f}]  box [{pr_lo[i]:.3f}, {pr_hi[i]:.3f}]")

# ---- Correlations; then fix M at its MAP value and refit gamma ----
print("\ncorrelation matrix of the samples:")
print(pd.DataFrame(np.corrcoef(mc_flat.T), index=mc_names, columns=mc_names).round(2))

mc_Mfix = mc_map[0]
def mc_logpost_Mfix(t4):
    return pr_logpost(np.concatenate([[mc_Mfix], t4]))

mc_nw2 = 24
mc_s2 = emcee.EnsembleSampler(mc_nw2, 4, mc_logpost_Mfix)
mc_c2 = mc_map[1:] + 0.1*mc_sig[1:]*mc_rng.normal(size=(mc_nw2, 4))
mc_s2.run_mcmc(mc_c2, 6000, progress=False)
mc_flat2 = mc_s2.get_chain(discard=2000, thin=10, flat=True)
mc_sg_marg = 0.5*np.subtract(*np.percentile(mc_flat[:, 3], [84, 16]))
mc_sg_fixM = 0.5*np.subtract(*np.percentile(mc_flat2[:, 2], [84, 16]))

# gamma-only (everything else at MAP): 1-D grid posterior
mc_gg = np.linspace(mc_map[3] - 6*mc_sig[3], mc_map[3] + 6*mc_sig[3], 801)
mc_lg = np.array([pr_logpost(np.concatenate([mc_map[:3], [g], mc_map[4:]])) for g in mc_gg])
mc_pg = np.exp(mc_lg - mc_lg.max()); mc_pg /= np.trapezoid(mc_pg, mc_gg)
mc_gm = np.trapezoid(mc_gg*mc_pg, mc_gg)
mc_sg_only = np.sqrt(np.trapezoid((mc_gg - mc_gm)**2*mc_pg, mc_gg))

print(f"\nsigma_gamma, marginal over all parameters : {mc_sg_marg:.4f}")
print(f"sigma_gamma, M fixed at MAP               : {mc_sg_fixM:.4f}   (ratio fixed/marginal = {mc_sg_fixM/mc_sg_marg:.3f})")
print(f"sigma_gamma, all others fixed at MAP      : {mc_sg_only:.4f}   (ratio = {mc_sg_only/mc_sg_marg:.3f})")
print(f"corr(M, gamma) = {np.corrcoef(mc_flat[:, 0], mc_flat[:, 3])[0, 1]:+.3f}, "
      f"sqrt(1-rho^2) = {np.sqrt(1 - np.corrcoef(mc_flat[:, 0], mc_flat[:, 3])[0, 1]**2):.3f}")

**ANSWER:**

Since we see that for all the chains we have $\hat{R}<1.05$, we can say with reasonable confidence that the MCMC was run for a long enough time. It's also worth comparing to the longest autocorrelation time, in my case 68 steps, so with a chain of length 10000, even after discarding roughly $9\tau$ steps for burn-in (600 to be exact), we easily have over $130\tau$ steps and should thus expect that our sampling ran for a long enough period. The most strongly correlated variables are $\gamma$ and $M$. $\gamma$ and $\alpha$ show some noticeable but not particularly strong correlation, as do $\beta$ and $\ln(\sigma_\mathrm{int})$. The correlation between $\gamma$ and $M$ is likely strong due to the Tripp relation; increase $M$ and you must decrease $\gamma$ to better fit the points where $\log(M_\star)$ is above the threshold and vice versa. The error on $\gamma$ we see with the fixed $M$ and with all parameters fixed are significantly lower than the marginal case as shown above, which makes sense as reducing the variability of other parameters should result in a narrower range of acceptable values for $\gamma$ that fit the data alongside the chosen fixed values.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here
md_N = int(lk_mask.sum())

def md_lnL(theta):
    return lk_loglike(theta, mask=lk_mask)

# M1 (gamma free): MAP and Hessian from 2c
md_th1 = mc_map.copy()
md_lnL1 = md_lnL(md_th1)
md_H1 = mc_hessian(mc_nlp, md_th1, np.array([5e-3, 5e-3, 2e-2, 5e-3, 2e-2]))
md_C1 = np.linalg.inv(md_H1)

# M0 (gamma = 0): refit the other four parameters
md_f0 = lambda t4: mc_nlp(np.insert(t4, 3, 0.0))
md_x0 = np.delete(mc_map, 3)
md_r0 = minimize(md_f0, md_x0, method='Nelder-Mead',
                 options={'xatol': 1e-9, 'fatol': 1e-12, 'maxiter': 20000, 'maxfev': 20000})
md_r0 = minimize(md_f0, md_r0.x, method='Nelder-Mead',
                 options={'xatol': 1e-9, 'fatol': 1e-12, 'maxiter': 20000, 'maxfev': 20000})
md_th0 = np.insert(md_r0.x, 3, 0.0)
md_lnL0 = md_lnL(md_th0)
md_H0 = mc_hessian(md_f0, md_r0.x, np.array([5e-3, 5e-3, 2e-2, 2e-2]))
md_C0 = np.linalg.inv(md_H0)

# Flat priors: shared parameters use the same box in both models, gamma gets a box of half-width g
md_w_shared = np.delete(pr_hi - pr_lo, 3)          # widths of M, alpha, beta, ln sigma_int

def md_2lnB(g):
    lnZ1 = (md_lnL1 - np.log(md_w_shared).sum() - np.log(2*g)
            + 0.5*5*np.log(2*np.pi) + 0.5*np.linalg.slogdet(md_C1)[1])
    lnZ0 = (md_lnL0 - np.log(md_w_shared).sum()
            + 0.5*4*np.log(2*np.pi) + 0.5*np.linalg.slogdet(md_C0)[1])
    return 2*(lnZ1 - lnZ0)

def md_strength(x):
    a = abs(x)
    s = 'weak' if a < 2 else 'positive' if a < 6 else 'strong' if a < 10 else 'very strong'
    return f"{s} for {'M1 (gamma free)' if x > 0 else 'M0 (gamma = 0)'}"

# 1. Laplace 2 ln B_10
MD_G = 0.15   # baseline prior
print(f"gamma prior U(-{MD_G}, {MD_G}):  2 ln B10 = {md_2lnB(MD_G):+.2f}  -> {md_strength(md_2lnB(MD_G))}")
print(f"twice as wide, U(-{2*MD_G}, {2*MD_G}):  2 ln B10 = {md_2lnB(2*MD_G):+.2f}  -> {md_strength(md_2lnB(2*MD_G))}")
print("sensitivity to the prior half-width:")
for md_g in [0.1, 0.3, 0.6, 1.0]:
    print(f"   +/-{md_g:<4}: 2 ln B10 = {md_2lnB(md_g):+.2f}   ({md_strength(md_2lnB(md_g))})")

# 2. BIC_0 - BIC_1 (positive favours M1)
md_dBIC = (4 - 5)*np.log(md_N) + 2*(md_lnL1 - md_lnL0)
print(f"\nBIC0 - BIC1 = {md_dBIC:+.2f}  -> {md_strength(md_dBIC)}")

# 3. gamma vs the published value
md_g_hat = mc_map[3]
md_sg = mc_sg_marg
print(f"\ngamma = {np.median(mc_flat[:, 3]):+.4f} +/- {md_sg:.4f}  (Laplace sigma_gamma = {np.sqrt(md_C1[3, 3]):.4f})")
print(f"gamma = 0 is {abs(md_g_hat)/md_sg:.1f} sigma away;  published -0.06 is {abs(-0.06 - md_g_hat)/md_sg:.1f} sigma from our gamma")
print(f"a -0.06 step would be {0.06/md_sg:.1f} sigma given our sigma_gamma")

**ANSWER:**

Here I chose a significantly lower prior for $\gamma$ than I did earlier, simply because I wanted to be absolutely certain my original MCMC was able to explore. If I were to have chosen a prior that I felt would be more representative of what I'd expect (before looking at the actual results, so we don't double count/bias things) I would have chosen between 0.1 and 0.2 mag, hence this prior. Even with a smaller prior though, the bounds are wide enough that the model shows (very) weak evidence for $\gamma=0$, which gets slightly stronger but still weak as we double the prior's width. $\mathrm{BIC}_0-\mathrm{BIC}_1=-2.07$ shows somewhat positive evidence for $\gamma=0$, though the published value lay at $3\sigma_\gamma$ from the supported model and only $1.2\sigma$ from the reported value of $\gamma$ so it's not conclusive evidence for $\gamma=0$. If I had to choose, I would say I think the published, non-zero value of $\gamma$ is more likely and that Bayes factors and the BIC just show that this dataset with 250 (now 246 after clipping) points does not necessarily affect the likelihood enough to justify adding the extra parameter, not that the parameter being 0 is a significantly better model.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here
TS_TRUTH = dict(M=-18.76, alpha=0.125, beta=3.14, gamma=-0.06, sig=0.09)
TS_NSIM  = 200
ts_rng   = np.random.default_rng(1234567890)

# --- sample quantities (contaminants removed) ---
ts_idx = np.where(lk_mask)[0]
ts_n   = len(ts_idx)
ts_mu, ts_x1t, ts_ct = sn_mu[ts_idx], lk_x1[ts_idx], lk_c[ts_idx]
ts_step, ts_pec2 = lk_step[ts_idx], lk_sig_pec2[ts_idx]
ts_Cmm, ts_Cxx, ts_Ccc = lk_Cmm[ts_idx], lk_Cxx[ts_idx], lk_Ccc[ts_idx]
ts_Cmx, ts_Cmc, ts_Cxc = lk_Cmx[ts_idx], lk_Cmc[ts_idx], lk_Cxc[ts_idx]

ts_C = np.zeros((ts_n, 3, 3))
ts_C[:, 0, 0], ts_C[:, 1, 1], ts_C[:, 2, 2] = ts_Cmm, ts_Cxx, ts_Ccc
ts_C[:, 0, 1] = ts_C[:, 1, 0] = ts_Cmx
ts_C[:, 0, 2] = ts_C[:, 2, 0] = ts_Cmc
ts_C[:, 1, 2] = ts_C[:, 2, 1] = ts_Cxc
ts_L = np.linalg.cholesky(ts_C)

ts_x1_mean, ts_x1_sd = ts_x1t.mean(), ts_x1t.std()
ts_c_mean,  ts_c_sd  = ts_ct.mean(),  ts_ct.std()
ts_truth = np.array([TS_TRUTH['M'], TS_TRUTH['alpha'], TS_TRUTH['beta'],
                     TS_TRUTH['gamma'], np.log(TS_TRUTH['sig'])])

# --- fake-data generators: each returns (m_B, x1_obs, c_obs) ---
def ts_mB_from(x1_lat, c_lat, e_m, extra_sd):
    t = TS_TRUTH
    return (t['M'] + ts_mu - t['alpha']*x1_lat + t['beta']*c_lat + t['gamma']*ts_step
            + e_m + ts_rng.normal(0, extra_sd))

def ts_make_first():            # latent = observed x1, c; fit noise added on top
    e = np.einsum('nij,nj->ni', ts_L, ts_rng.normal(size=(ts_n, 3)))
    sd = np.sqrt(ts_pec2 + TS_TRUTH['sig']**2)
    return ts_mB_from(ts_x1t, ts_ct, e[:, 0], sd), ts_x1t + e[:, 1], ts_ct + e[:, 2]

def ts_make_second(k=10):            # latent population k x wider than observed, full correlated noise
    x1_lat = ts_x1_mean + k*ts_x1_sd*ts_rng.normal(size=ts_n)
    c_lat  = ts_c_mean  + k*ts_c_sd *ts_rng.normal(size=ts_n)
    e = np.einsum('nij,nj->ni', ts_L, ts_rng.normal(size=(ts_n, 3)))
    sd = np.sqrt(ts_pec2 + TS_TRUTH['sig']**2)
    return ts_mB_from(x1_lat, c_lat, e[:, 0], sd), x1_lat + e[:, 1], c_lat + e[:, 2]

# --- the 2b likelihood on the fake data, and the fitting loop ---
def ts_nll(theta, mB, x1, c):
    M, al, be, ga, lns = theta
    r = mB - ts_mu - M + al*x1 - be*c - ga*ts_step
    var = (ts_Cmm + al**2*ts_Cxx + be**2*ts_Ccc + 2*al*ts_Cmx - 2*be*ts_Cmc - 2*al*be*ts_Cxc
           + ts_pec2 + np.exp(2*lns))
    return 0.5*np.sum(r**2/var + np.log(var) + np.log(2*np.pi))

ts_h = np.array([5e-3, 5e-3, 2e-2, 5e-3, 2e-2])
def ts_run(make):
    est, err = [], []
    for _ in range(TS_NSIM):
        mB, x1, c = make()
        A  = np.column_stack([np.ones(ts_n), -x1, c, ts_step])
        s0 = np.append(np.linalg.lstsq(A, mB - ts_mu, rcond=None)[0], np.log(0.1))
        r  = minimize(ts_nll, s0, args=(mB, x1, c), method='Nelder-Mead',
                      options={'xatol': 1e-8, 'fatol': 1e-10, 'maxiter': 10000, 'maxfev': 10000})
        r  = minimize(ts_nll, r.x, args=(mB, x1, c), method='Nelder-Mead',
                      options={'xatol': 1e-9, 'fatol': 1e-12, 'maxiter': 10000, 'maxfev': 10000})
        try:
            sd = np.sqrt(np.diag(np.linalg.inv(mc_hessian(lambda t: ts_nll(t, mB, x1, c), r.x, ts_h))))
        except np.linalg.LinAlgError:
            sd = np.full(5, np.nan)
        est.append(r.x); err.append(sd)
    return np.array(est), np.array(err)

# --- run the two scenarios ---
ts_scen = {
    'First test (INCORRECT: adds noise to already-noisy observations)':
        ts_make_first,
    'Coverage test (CORRECT: model-matched latent population, k = 10)':
        lambda: ts_make_second(10)
}

ts_res = {}
for ts_name, ts_mk in ts_scen.items():
    ts_e, ts_s = ts_run(ts_mk)
    ts_okm = np.all(np.isfinite(ts_s), axis=1)
    ts_e, ts_s = ts_e[ts_okm], ts_s[ts_okm]
    ts_res[ts_name] = (ts_e, ts_s, np.abs((ts_e - ts_truth)/ts_s) < 1)

# --- coverage summary ---
ts_sec = np.sqrt(0.68 * 0.32 / TS_NSIM)

print(f"{'coverage of 68% intervals':<60}"
      + "".join(f"{n:>9}" for n in ['M', 'alpha', 'beta', 'gamma', 'ln sig'])
      + "   verdict")

for ts_name, (ts_e, ts_s, ts_hit) in ts_res.items():
    ts_c = ts_hit.mean(axis=0)

    # keep pass/fail criterion at ±1 sigma_binom
    ts_v = 'pass' if np.all(np.abs(ts_c - 0.68) < 2 * ts_sec) else 'FAIL'

    print(
        f"{ts_name:<60}"
        + "".join(f"{x:>9.3f}" for x in ts_c)
        + f"   {ts_v}"
    )

print(f"(pass = every coverage within 0.68 +/- 2 x {ts_sec:.3f})")

# --- plots ---
ts_plabels = [r'$M$', r'$\alpha$', r'$\beta$', r'$\gamma$', r'$\ln\sigma_{\rm int}$']
ts_cols = ['C3', 'C2']

fig, ts_ax = plt.subplots(
    1, 2,
    figsize=(15, 4.8),
    gridspec_kw={'width_ratios': [1.1, 1]}
)

ts_a = ts_ax[0]
ts_w = 0.35

for ts_j, (ts_name, (ts_e, ts_s, ts_hit)) in enumerate(ts_res.items()):
    ts_c = ts_hit.mean(axis=0)

    ts_a.bar(
        np.arange(5) + (ts_j - 0.5) * ts_w,
        ts_c,
        ts_w,
        yerr=np.sqrt(ts_c * (1 - ts_c) / len(ts_hit)),
        capsize=3,
        color=ts_cols[ts_j],
        alpha=0.8,
        label=ts_name
    )
ts_a.axhline(
    0.68,
    color='k',
    ls='--',
    lw=1
)
# ±1σ binomial band
ts_a.axhspan(
    0.68 - 2 * ts_sec,
    0.68 + 2 * ts_sec,
    color='gray',
    alpha=0.20,
    label=r'$0.68 \pm 2\sigma_{\rm binom}$'
)

ts_a.set_xticks(np.arange(5))
ts_a.set_xticklabels(ts_plabels)
ts_a.set_ylabel(r'fraction of $68\%$ intervals containing the truth')
ts_a.set_ylim(0, 1)
ts_a.legend(fontsize=8, loc='upper left')

ts_a = ts_ax[1]

ts_e, ts_s, ts_hit = ts_res[
    'Coverage test (CORRECT: model-matched latent population, k = 10)'
]

ts_np = min(60, len(ts_e))
ts_ib = 2  # beta
for ts_k in range(ts_np):
    ts_a.errorbar(
        ts_e[ts_k, ts_ib] - ts_truth[ts_ib],
        ts_k,
        xerr=ts_s[ts_k, ts_ib],
        fmt='o',
        ms=3,
        color='C2' if ts_hit[ts_k, ts_ib] else 'C3',
        lw=1
    )
ts_a.axvline(
    0,
    color='k',
    lw=1,
    label='truth'
)

# FULL coverage statistics (not just first 60 plotted points)
ts_pass = ts_hit[:, ts_ib].sum()
ts_total = len(ts_hit)
ts_pct = 100 * ts_pass / ts_total

ts_a.set_xlabel(r'$\hat{\beta}-\beta_{\rm true}$')
ts_a.set_ylabel('simulated dataset')

ts_a.set_title(
    rf'Correct test, $\beta$: '
    rf'{ts_pass}/{ts_total} intervals contain truth '
    rf'({ts_pct:.1f}%)'
)

ts_a.legend(fontsize=8)
plt.tight_layout()
plt.show()

**ANSWER:**

The test plan is as suggested. We'll simulate 300 fake datasets at the real redshifts, host masses, $x_1$ and $c$ of the supernovae kept after the contaminant cut. For this I've chosen the truth values to be $M=-18.76$, $\alpha=0.125$, $\beta=3.14$, $\gamma=-0.06$, $\sigma_{\rm int}=0.09$ mag. Then we just refit with maximum likelihood and use a Hessian errorbar.

The failure criteria for this test is simple: if we get back values which have significantly inflated/deflated errors s.t. coverage does not lie near 68% for all variables, we know we have an issue.

Note that here the red case on the left plot was from a mistake made by Claude when simulating, where extra error was added to the observations which made my tests fail. I only decided to keep that case in as it's a good demonstration of an error I will report in 3 below, and because I already had the plot made for debugging to see if the fix worked. 

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
### Setup code, running the AI solution without plots/prints
d = np.genfromtxt(f'data/{NETID}/p3_wd_masses.csv', delimiter=',', names=True, dtype=None, encoding='utf-8')
mass = d['mass_Msun']
med, sd = np.median(mass), np.std(mass)
keep = np.abs(mass - med) < 3 * sd
m = mass[keep]

def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])

np.random.seed(42)
nwalk, ndim = 32, 5
start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(m,))
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain(discard=500)              # (steps, walkers, params)
#print(sampler.get_autocorr_time())
flat = chain.reshape(-1, ndim)

def gelman_rubin(x):                                  # x: (steps, chains)
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

In [ ]:
### Demonstration 1
d = np.genfromtxt(
    f"data/{NETID}/p3_wd_masses.csv",
    delimiter=",",
    names=True,
    dtype=None,
    encoding=None,
)

mass = d["mass_Msun"]

# AI solution clipping
med = np.median(mass)
sd = np.std(mass)

keep = np.abs(mass - med) < 3 * sd
removed = mass[~keep]

print(f"Original sample: {len(mass)} stars")
print(f"Stars removed by clipping: {len(removed)}")

print("\nRemoved masses:")
print(np.sort(removed))

print(f"\nMaximum mass in dataset : {mass.max():.3f}")
print(f"Maximum mass kept       : {mass[keep].max():.3f}")

bins = np.linspace(mass.min(), mass.max(), 50)
plt.figure(figsize=(7,4))
plt.hist(mass, bins=bins, alpha=0.6, label="All stars")
plt.hist(removed, bins=bins, color="red", label="Removed by clipping")
plt.xlabel("Mass (Msun)")
plt.ylabel("Count")
plt.title("Demonstration 1: Sigma-clipping removes stars")
plt.legend()
plt.show()

print('If anything, I\'d be more curious about whether the lowest mass cases are outliers rather than the highest mass cases,')
print('as they seem less like part of either distribution.')

In [ ]:
### Demonstration 2, direct from the AI solution
names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
best = np.median(flat, axis=0)
rng = np.random.default_rng(1)
n2 = rng.binomial(m.size, 1 - best[4])
sim = np.concatenate([rng.normal(best[0], best[1], m.size - n2), rng.normal(best[2], best[3], n2)])
vs = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(sim,))
vs.run_mcmc(best + 1e-3 * np.random.randn(nwalk, ndim), 2000, progress=False)
rec = np.median(vs.get_chain(discard=500, flat=True), axis=0)
for nm, a, b in zip(names, best, rec):
    print(f"{nm}: input {a:.3f}, recovered {b:.3f} ({100 * (b - a) / a:+.1f}%)")
print('\nAI decided "Input parameters recovered: the pipeline is validated", even with >20% error on the returned value of sigma_1.')

In [ ]:
### Demonstration 3
# From AI code directly
mu1, s1, mu2, s2, f = best
one = stats.norm.fit(m)
lnL1 = np.sum(stats.norm.logpdf(m, *one)); lnL2 = log_prob(best, m)

# New code
def log_prob_3(p, m):
    mu1, s1, mu2, s2, mu3, s3, f1, f2 = p

    if not (
        0.3 < mu1 < 1.3 and
        0.3 < mu2 < 1.3 and
        0.3 < mu3 < 1.3 and
        0.005 < s1 < 0.5 and
        0.005 < s2 < 0.5 and
        0.005 < s3 < 0.5 and
        0 < f1 < 1 and
        0 < f2 < 1 and
        f1 + f2 < 1
    ):
        return -np.inf

    pdf = (
        f1 * stats.norm.pdf(m, mu1, s1) +
        f2 * stats.norm.pdf(m, mu2, s2) +
        (1 - f1 - f2) * stats.norm.pdf(m, mu3, s3)
    )

    return np.sum(np.log(pdf))

# initialize from quantiles
q1, q2 = np.percentile(m, [33, 66])

g1 = m[m < q1]
g2 = m[(m >= q1) & (m < q2)]
g3 = m[m >= q2]

p0_3 = np.array([
    g1.mean(), g1.std(),
    g2.mean(), g2.std(),
    g3.mean(), g3.std(),
    len(g1)/len(m),
    len(g2)/len(m)
])

nwalk = 40
ndim = len(p0_3)

start = p0_3 + 1e-3*np.random.randn(nwalk, ndim)

sampler3 = emcee.EnsembleSampler(
    nwalk,
    ndim,
    log_prob_3,
    args=(m,)
)

sampler3.run_mcmc(start, 5000, progress=True)

flat3 = sampler3.get_chain(discard=500, flat=True)
best3 = np.median(flat3, axis=0)

lnL3 = log_prob_3(best3, m)

def log_prob_4(p, m):
    mu1,s1,mu2,s2,mu3,s3,mu4,s4,f1,f2,f3 = p

    if not (
        all(0.3 < x < 1.3 for x in [mu1,mu2,mu3,mu4]) and
        all(0.005 < x < 0.5 for x in [s1,s2,s3,s4]) and
        all(0 < x < 1 for x in [f1,f2,f3]) and
        f1 + f2 + f3 < 1
    ):
        return -np.inf

    pdf = (
        f1*stats.norm.pdf(m, mu1, s1) +
        f2*stats.norm.pdf(m, mu2, s2) +
        f3*stats.norm.pdf(m, mu3, s3) +
        (1-f1-f2-f3)*stats.norm.pdf(m, mu4, s4)
    )

    return np.sum(np.log(pdf))

q = np.percentile(m, [25, 50, 75])

groups = [
    m[m < q[0]],
    m[(m >= q[0]) & (m < q[1])],
    m[(m >= q[1]) & (m < q[2])],
    m[m >= q[2]]
]

p0_4 = np.array([
    groups[0].mean(), groups[0].std(),
    groups[1].mean(), groups[1].std(),
    groups[2].mean(), groups[2].std(),
    groups[3].mean(), groups[3].std(),
    len(groups[0])/len(m),
    len(groups[1])/len(m),
    len(groups[2])/len(m)
])

nwalk = 48
ndim = len(p0_4)

start = p0_4 + 1e-3*np.random.randn(nwalk, ndim)

sampler4 = emcee.EnsembleSampler(
    nwalk,
    ndim,
    log_prob_4,
    args=(m,)
)

sampler4.run_mcmc(start, 5000, progress=True)

flat4 = sampler4.get_chain(discard=500, flat=True)
best4 = np.median(flat4, axis=0)

lnL4 = log_prob_4(best4, m)

print(f"1 population logL = {lnL1:.1f}")
print(f"2 population logL = {lnL2:.1f}")
print(f"3 population logL = {lnL3:.1f}")
print(f"4 population logL = {lnL4:.1f}")
print('Adding populations increases the log-likelihood, so just reporting that without further analysis is insufficient.')

**REFEREE REPORT:**

Error 1: Sigma-clipping the outliers. First of all, even using std rather than something like MAD_std to find the standard deviation for clipping here is wrong as that will artificially inflate the deviation. However, the main issue is the clipping itself. The AI doesn't check whether they're actually likely to be outliers, just rejecting anything outside an already wrong std. Since the problem is entirely based on the idea of having a second population of stars which is significantly more massive than you would expect, clipping here is highly likely to remove completely valid data points. Even if it only removes a few that can still be an issue and potentially waste hours or maybe days of sensor time. See demonstration 1.

Error 2: Its own validation failed, and the AI still automatically assumes it had passed with no issue. See demonstration 2, it's a very simple check: just printing "verification passed" without checking that the values get back are reasonable is a bad idea.

Error 3: The AI used the likelihoods to compare the models rather than Bayes factors or BIC. Likelihoods will almost always increase if you add a parameter (in this case, a second gaussian population) to the fit, just as we had seen in class. I still think the result is likely correct, that there are in fact two populations, but regardless of the outcome of the AI solution, the check here was not a valid confirmation of the two-population model. See demonstration 3.

There's at least one more (checking autocorrelation times shows the MCMC likely didn't run long enough even despite the nice $\hat{R}$ values) but I'll only detail/demonstrate the three above.

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

1. All parts (besides 3b here) I used Claude to generate the code. In some instances, specifically for the demonstrations in 3a, I used Copilot instead by providing the AI solution, my write-up, and the demonstrations I wanted it to produce. Apparently either I wasn't that clear when prompting Copilot, or it was strangely ineffective in understanding what was intended, as it messed up the last demonstration in two major ways. I told it to compare the likelihoods computed by the AI with the likelihoods from 3- and 4-population models of the data and it first decided to attempt it on synthetic data without even doing MCMC (instead using sklearn GaussianMixture) as I thought was implied by the prior code. Needless to say I quickly caught that and had it fixed. Other than Copilot's error, the other code segments from Claude usually had no issue, the one exception being the verification code for 2e which still shows the original results as I thought it would be useful for demonstrating its error in this section. Basically Claude originally made an error by adding additional scatter on the observed data ($x_1$, $c$) by treating it as latent truths instead.

2. Since I described two errors I found, those could have caused issues. Not checking Copilot would have led to a demonstration that didn't really demonstrate anything relevant to the AI's solution. It would have been comparing apples to oranges, and even if the idea behind the caught error was right the demonstration would not have been applied to the actual data or be representative of the real MCMC 3/4 population model results. In the case with Claude's error, I could've been stuck assuming that the likelihood model we used throughout 2 was significantly incorrect when the issue was actually with the sim code. There would have also been several instances where the results were *technically* fine, but just not ideal: mostly cases of claude oversharing and printing several tables with information which I did not ask for and which was (usually) irrelevant or only somewhat relevant to what was needed for the problem, or cases of plots being a little badly designed (such as Claude deciding to use different bins on the clipping demonstration, so it was hard to tell what exactly had been clipped).

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.